### Colab / Drive helper

If you run this notebook on Google Colab, run the cell below to mount your Google Drive and optionally set the path to your repository copy (so the repo root is available). If not running on Colab the cell will simply print a message and continue.

# Notebook adapted: 2-layer GE diagnostics and power-law runs

This notebook runs two types of experiments using the project's `run_*.py` runners:
1. Gaussian-equivalence diagnostic: compute NMSE and spectra for a sweep of alphas, comparing the true teacher and the gaussian-equivalent (GE) model.
2. Power-law experiment: runs the same diagnostics with `gamma>0` (power-law) to inspect effect of coordinate scaling on the first layer.

Note that, it should be possible to use other g* or k=3, but several checks need to be done. I will ensure to make it works in the next version

In [ ]:
# Colab mount helper (run this only if on Colab)
import sys
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ## !!! ADJUST THE FOLLOWING PATH TO POINT TO YOUR REPO ROOT ON THE MOUNTED DRIVE !!! ##
    DRIVE_REPO_PATH = '/content/drive/MyDrive/Colab Notebooks/hierarchical-model'
    repo_root = Path(DRIVE_REPO_PATH)
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    print('Running in Colab; mounted drive and added repo_root:', repo_root)
else:
    print('Not running in Colab; notebook will use local repo root')

In [ ]:
# 1) Setup imports and path
import os
import sys
from pathlib import Path
# ensure repository root is on path
repo_root = Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import numpy as np
import run_2layers_nmse as nmse_run
import run_2layers_spectrum as spectrum_run
import teacher
import estimators
import measures as mps

print('imports ok, torch device:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

In [ ]:
# 2) Common parameters (tweak these)

ds = [40, 60, 80]
eps = 0.5   # will set n = round(d**alpha) in the exponential regime and n = alpha_prop d**(2+eps) in the proportional regime
reps = 10   # number of repetitions for each setting (will average results over these)   
beta = 1.0      # Factor in the definition of B, never used, should be removed in the next version 
n_test = 10000      # number of test samples 
batch_size = 1024       # batch size for training
n_cap = 1e12            # limit size for training samples, don't like it personnally so super high. 
A_mode_teacher = 'sym_orth_frob'        # Previous version, there was possibility of rank 1 for A, should be removed in the next version.
n_iter_C = 15               # number of iterations in power iterations method (computation of C1)
oversamp_C = 10             # oversampling factor for power iterations method (computation of C1)   
seed0 = 0                   # random seed
g_name = 'id'               # non-linearity g*
gamma = None                # power-law exponent
models  = ['true', 'gauss']         # models to run, gauss stand for the GE model, 


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Adjust this path to where you want to save results, note that I am not saving in the same repo to avoid cluttering, but you can adjust as you want.
out_base = Path('/content/drive/MyDrive/notebook_march_runs_1').resolve()
out_base.mkdir(parents=True, exist_ok=True)
print('params set; out_base=', out_base)

## Experiment A: Gaussian-equivalence diagnostic (NMSE porportionnal regime + NMSE exponential regime + Spectra)

We run the existing runners with g = id (default) and gamma = None. The NMSE runner will run both 'true' and 'gauss' models and save results; the spectra runner saves eigenvalues for each setting.

Adjust `alphas` above before executing.

In [ ]:
# 3) Run NMSE sweep (this may take time depending on parameters)
alphas_prop = np.linspace(0.01, 100, 20)

for d in ds:

    alphas = np.log(alphas_prop) / np.log(d) + 2 # Setting the proportional regime through the exponent, not optimal but fine. 

    out_dir_nmse = out_base / f'exp_GE_nmse_d{d}'
    out_dir_nmse.mkdir(parents=True, exist_ok=True)
    res_nmse = nmse_run.run_seq_sweep_alpha_using_your_C(
        d=d, eps=eps, alphas=alphas, reps=reps, beta=beta, n_test=n_test,
        batch_size=batch_size, n_cap=n_cap, A_mode_teacher=A_mode_teacher,
        n_iter_C=n_iter_C, oversamp_C=oversamp_C, path=str(out_dir_nmse / 'overlaps_propregime'),
        seed0=seed0, device=device, gamma=gamma, g_name=g_name, g_callable=None,
        models=models
    )
    print('NMSE run finished; results keys:', list(res_nmse.keys()))

In [ ]:
# 4) Run NMSE sweep (this may take time depending on parameters)
alphas = list(np.arange(0.2, 4.2, 0.2))  # will set n = round(d**alpha)

for d in ds:
    out_dir_nmse = out_base / f'exp_GE_nmse_d{d}'
    out_dir_nmse.mkdir(parents=True, exist_ok=True)
    res_nmse = nmse_run.run_seq_sweep_alpha_using_your_C(
        d=d, eps=eps, alphas=alphas, reps=reps, beta=beta, n_test=n_test,
        batch_size=batch_size, n_cap=n_cap, A_mode_teacher=A_mode_teacher,
        n_iter_C=n_iter_C, oversamp_C=oversamp_C, path=str(out_dir_nmse / 'overlaps'),
        seed0=seed0, device=device, gamma=gamma, g_name=g_name, g_callable=None,
        models=models
    )
    print('NMSE run finished; results keys:', list(res_nmse.keys()))

In [ ]:
# 5) Run spectra diagnostics (L1 and L2 for true vs GE) (not ready yet)
# for d in ds:
#     out_dir_spec = out_base / f'exp_GE_spectra_d{d}'
#     out_dir_spec.mkdir(parents=True, exist_ok=True)
#     # we choose a single alpha (n is passed directly) - loop over alphas if you want multiple files
#     for alpha in alphas:
#         n = int(round(d ** alpha))
#         print(f'Computing spectra for alpha={alpha} (n={n})')
#         specs = spectrum_run.run_2layers_spectra_per_alpha(
#             d=d, p=int(round(d**eps)), n=n, batch_size=batch_size,
#             A_mode=A_mode_teacher, beta=beta, seed=seed0, device=device,
#             out_dir=str(out_dir_spec / f'alpha_{alpha}'), do_tau=False, n_test=1000,
#             g_name=None, g_callable=None, gamma=None
#         )
#         print('saved spectra for alpha=', alpha)

#     print('Spectra diagnostics finished')

## Experiment B: Power-law (gamma) runs

This cell runs the NMSE diagnostics with non-zero `gamma` values (power-law scaling on first-layer features). We keep g = id and run only the true/standard model for these tests. Results are saved under `./notebook_runs/exp_powerlaw/`.

In [ ]:
# 6) Power-law sweep (gamma) - separated into its own cell
gamma_list = list(np.arange(0., 1.0, 0.2))
alphas = list(np.arange(1.5, 3.5, 0.01)) 
models = ['true']  # no GE 
B_mode = 'powerlaw_diag'  # ensure power law!!

for d in ds:
    out_dir_gamma = out_base / f'exp_powerlaw_d{d}'
    out_dir_gamma.mkdir(parents=True, exist_ok=True)

    for gamma in gamma_list:
        print('Running power-law experiment with gamma=', gamma)
        
        res = nmse_run.run_seq_sweep_alpha_using_your_C(
            d=d, eps=eps, alphas=alphas, reps=reps, beta=beta, n_test=n_test,
            batch_size=batch_size, n_cap=n_cap, A_mode_teacher=A_mode_teacher,
            n_iter_C=n_iter_C, oversamp_C=oversamp_C, path=str(out_dir_gamma / f'overlaps_gamma_{gamma}'),
            seed0=seed0, device=device, g_name=g_name, g_callable=None,
            models=models, B_mode=B_mode, gamma=gamma,
        )
        print('done gamma=', gamma)

    print('Power-law experiments finished')